<div align="center">

# Laboratorio No. 1

## Preparación de un corpus y Análisis Exploratorio de Datos

<br>

**Curso:** Natual Language Processing
**Estudiante:** Jose Tánchez  
**Carné:** 20230005  

<br>

---

### Exploración, preparación y análisis del corpus  
### Spanish News Classification

</div>

# 2. Exploración inicial del corpus

Antes de aplicar cualquier proceso de normalización, se realizará una exploración de los datos para conocer varias cosas.

> En esta sección el corpus únicamente será explorado. No se modificarán ni eliminarán datos.

In [4]:
import pandas as pd
from pathlib import Path
from IPython.display import display

archivos_csv = list(Path(".").glob("*.csv"))

if not archivos_csv:
    raise FileNotFoundError(
        "No se encontró ningún archivo CSV. "
        "Coloca el corpus en la misma carpeta que el notebook."
    )

ruta_corpus = archivos_csv[0]

try:
    df = pd.read_csv(ruta_corpus, sep=None, engine="python")
except UnicodeDecodeError:
    df = pd.read_csv(
        ruta_corpus,
        sep=None,
        engine="python",
        encoding="latin-1"
    )

print(f"Archivo cargado: {ruta_corpus.name}")
print(f"Dimensiones del corpus: {df.shape}")

display(df.head())

Archivo cargado: df_total.csv
Dimensiones del corpus: (1217, 3)


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


In [5]:
cantidad_documentos = df.shape[0]

print(f"Cantidad total de documentos: {cantidad_documentos:,}")

Cantidad total de documentos: 1,217


## 2.1 Cantidad de documentos

El corpus contiene un total de **[1217] documentos**.

Para este análisis, cada fila del conjunto de datos se considera un documento o noticia individual.

In [6]:
resumen_columnas = pd.DataFrame({
    "Columna": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values,
    "Valores no vacíos": df.notna().sum().values,
    "Valores vacíos": df.isna().sum().values,
    "Ejemplo": [
        df[columna].dropna().iloc[0]
        if not df[columna].dropna().empty
        else "Sin datos"
        for columna in df.columns
    ]
})

print(f"El corpus contiene {len(df.columns)} columnas.")

display(resumen_columnas)

El corpus contiene 3 columnas.


,Columna,Tipo de dato,Valores no vacíos,Valores vacíos,Ejemplo
0,url,str,1217,0,https://www.larepublica.co/redirect/post/3201905
1,news,str,1217,0,Durante el foro La banca articulador empresari...
2,Type,str,1217,0,Otra


## 2.2 Columnas del corpus

El conjunto de datos contiene **[3] columnas**:

| Columna | Información almacenada |
|---|---|
| `[URL]` | [Esta columan contiene la url de donde salio la noticia] |
| `[NEWS]` | [Esta columna contiene la noticia como tal] |
| `[TYPE]` | [Esta columna contiene el tipo de noticia de la que se trata] |

Cada fila representa una noticia, mientras que las columnas almacenan sus diferentes características, como el texto, el título o la categoría a la que pertenece.

In [7]:
distribucion_categorias = df["Type"].value_counts().reset_index()
distribucion_categorias.columns = ["Categoría", "Cantidad de noticias"]

distribucion_categorias["Porcentaje"] = (
    distribucion_categorias["Cantidad de noticias"] / len(df) * 100
).round(2)

print(f"Cantidad total de categorías: {df['Type'].nunique()}")

display(distribucion_categorias)

Cantidad total de categorías: 7


,Categoría,Cantidad de noticias,Porcentaje
0,Macroeconomia,340,27.94
1,Alianzas,247,20.30
2,Innovacion,195,16.02
3,Regulaciones,142,11.67
4,Sostenibilidad,137,11.26
5,Otra,130,10.68
6,Reputacion,26,2.14


## 2.3 Categorías del corpus

La columna **`Type`** indica la categoría a la que pertenece cada noticia.

El corpus contiene un total de **[7] categorías diferentes**. La distribución de las noticias por categoría es la siguiente:

| Categoría | Cantidad de noticias | Porcentaje |
|---|---:|---:|
| [Macroeconomia] | [340] | [27.94]% |
| [Alianzas] | [247] | [20.3]% |
| [Inovacion] | [195] | [16.02]% |
| [Regulaciones] | [142] | [11.67]% |
| [Sostenibilidad] | [137] | [11.26]% |
| [Otra] | [130] | [10.68]% |
| [Reputacion] | [26] | [2.14]% |

La categoría con más noticias es **[Macroeconomia]**, con **[340] documentos**, mientras que la categoría con menos noticias es **[Reputacion]**, con **[26] documentos**.

Esta distribución permite observar si las categorías del corpus se encuentran equilibradas o si algunas contienen muchas más noticias que otras.

In [8]:
# Crear una copia temporal para considerar los textos vacíos como valores nulos
df_revision = df.replace(r"^\s*$", pd.NA, regex=True)

# Revisar valores vacíos
filas_con_vacios = df_revision.isna().any(axis=1).sum()
filas_completamente_vacias = df_revision.isna().all(axis=1).sum()

vacios_por_columna = df_revision.isna().sum()

# Revisar duplicados
filas_duplicadas_completas = df.duplicated().sum()
urls_duplicadas = df["url"].duplicated().sum()
noticias_duplicadas = df["news"].duplicated().sum()

# Los valores repetidos en Type son normales porque representan categorías
tipos_repetidos = df["Type"].duplicated().sum()

resumen_calidad = pd.DataFrame({
    "Verificación": [
        "Filas con al menos un valor vacío",
        "Filas completamente vacías",
        "Filas completamente duplicadas",
        "URL duplicadas",
        "Noticias duplicadas",
        "Valores repetidos en Type"
    ],
    "Cantidad": [
        filas_con_vacios,
        filas_completamente_vacias,
        filas_duplicadas_completas,
        urls_duplicadas,
        noticias_duplicadas,
        tipos_repetidos
    ]
})

print("Valores vacíos por columna:")
display(vacios_por_columna.to_frame(name="Cantidad de valores vacíos"))

print("Resumen de valores vacíos y duplicados:")
display(resumen_calidad)

Valores vacíos por columna:


,Cantidad de valores vacíos
url,0
news,4
Type,0


Resumen de valores vacíos y duplicados:


,Verificación,Cantidad
0,Filas con al menos un valor vacío,4
1,Filas completamente vacías,0
2,Filas completamente duplicadas,75
3,URL duplicadas,116
4,Noticias duplicadas,79
5,Valores repetidos en Type,1210


## 2.4 Valores vacíos y duplicados

Para evaluar la calidad del corpus, se revisaron los valores vacíos y las duplicaciones presentes en cada una de sus columnas.

### Valores vacíos

| Verificación | Cantidad |
|---|---:|
| Filas con al menos un valor vacío | **[4]** |
| Filas completamente vacías | **[0]** |

La distribución de valores vacíos por columna fue la siguiente:

| Columna | Valores vacíos |
|---|---:|
| `URL` | [0] |
| `News` | [4] |
| `Type` | [0] |

### Valores duplicados

| Verificación | Cantidad |
|---|---:|
| Filas completamente duplicadas | **[75]** |
| URL duplicadas | **[116]** |
| Noticias duplicadas | **[79]** |

Las **filas completamente duplicadas** son aquellas en las que los valores de `URL`, `News` y `Type` son iguales.

Las **URL duplicadas** indican que una misma dirección aparece más de una vez, mientras que las **noticias duplicadas** representan contenidos textuales repetidos dentro del corpus.

La columna `Type` también contiene valores repetidos, pero esto es normal, ya que varias noticias pueden pertenecer a una misma categoría. Por esta razón, sus repeticiones no se consideran un problema de duplicación.

En esta etapa únicamente se identificaron los posibles problemas del corpus. Todavía no se eliminaron filas ni se modificaron los datos.

# 3. Preparación del corpus

En esta sección se aplicará un proceso de normalización sobre el contenido de las noticias.

El procedimiento se realizará en el siguiente orden:

1. **Tokenización:** separación del texto en unidades individuales.
2. **Conversión a minúsculas:** unificación de palabras escritas con diferentes combinaciones de mayúsculas y minúsculas.
3. **Eliminación de puntuación:** eliminación de puntos, comas, signos de interrogación y otros símbolos.
4. **Eliminación de stopwords:** eliminación de palabras frecuentes que aportan poca información al análisis.
5. **Lematización:** reducción de las palabras a su forma base o forma de diccionario.

> Después de cada etapa se calculará la cantidad total de **tokens** y la cantidad de **tipos**, es decir, palabras diferentes.

In [ ]:
import re
import nltk
import spacy
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from spacy.tokens import Doc
from IPython.display import display


nlp = spacy.load(
    "es_core_news_sm",
    disable=["parser", "ner"]
)

print("Librerías y modelo de español cargados correctamente.")

Librerías y modelo de español cargados correctamente.


In [11]:
# Crear una copia del corpus original
corpus = df.copy()

# Reemplazar posibles valores vacíos por texto vacío
corpus["news"] = corpus["news"].fillna("").astype(str)

print(f"Cantidad de documentos que serán procesados: {len(corpus):,}")

display(corpus[["url", "news", "Type"]].head())

Cantidad de documentos que serán procesados: 1,217


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


In [12]:
# Diccionario donde se almacenarán los resultados
resultados_normalizacion = {}


def contar_tokens_y_tipos(columna_tokens):
    """
    Cuenta la cantidad total de tokens y tipos
    presentes en una columna de listas.
    """

    cantidad_tokens = 0
    vocabulario = set()

    for documento in columna_tokens:
        cantidad_tokens += len(documento)
        vocabulario.update(documento)

    cantidad_tipos = len(vocabulario)

    return cantidad_tokens, cantidad_tipos


def registrar_resultado(etapa, columna_tokens):
    """
    Calcula y guarda la cantidad de tokens y tipos
    obtenidos en una etapa del proceso.
    """

    tokens, tipos = contar_tokens_y_tipos(columna_tokens)

    resultados_normalizacion[etapa] = {
        "Tokens": tokens,
        "Tipos": tipos
    }

    print(f"Etapa: {etapa}")
    print(f"Tokens totales: {tokens:,}")
    print(f"Tipos diferentes: {tipos:,}")

## 3.1 Tokenización

La tokenización consiste en dividir cada noticia en unidades más pequeñas llamadas **tokens**.

En esta primera etapa se conservan las palabras, los números y los signos de puntuación tal como aparecen en el texto original.

In [13]:
# Dividir cada noticia en tokens
corpus["tokens_tokenizados"] = corpus["news"].apply(
    lambda texto: word_tokenize(
        texto,
        language="spanish"
    )
)

# Contar tokens y tipos
registrar_resultado(
    "1. Tokenización",
    corpus["tokens_tokenizados"]
)

# Mostrar un ejemplo
print("\nPrimeros 40 tokens de la primera noticia:")
print(corpus["tokens_tokenizados"].iloc[0][:40])

Etapa: 1. Tokenización
Tokens totales: 665,592
Tipos diferentes: 38,947

Primeros 40 tokens de la primera noticia:
['Durante', 'el', 'foro', 'La', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el', 'director', 'de', 'sostenibilidad', 'y', 'clientes', 'globales', 'de', 'BBVA', 'en', 'Colombia', 'Andrés', 'García', 'aseguró', 'que', 'es', 'importante', 'entender', 'que', 'la', 'sostenibilidad', 'no', 'la', 'podemos', 'asociar', 'a', 'mayores', 'costos', '.']


### Resultado de la tokenización

Después de tokenizar todas las noticias se obtuvieron:

- **Tokens totales:** `[665592]`
- **Tipos diferentes:** `[38947]`

En esta etapa todavía se conservan las mayúsculas, las minúsculas y los signos de puntuación. Por esta razón, palabras como `Hola` y `hola` todavía son consideradas tipos diferentes.

## 3.2 Conversión a minúsculas

En esta etapa todos los tokens se convierten a minúsculas.

Esto permite unificar palabras que tienen el mismo significado, pero que se encuentran escritas de manera diferente debido al uso de mayúsculas.

Por ejemplo, `Hola`, `HOLA` y `hola` pasan a ser el mismo tipo: `hola`.

In [ ]:
# Convertir cada token a minúsculas
corpus["tokens_minusculas"] = corpus["tokens_tokenizados"].apply(
    lambda tokens: [
        token.lower()
        for token in tokens
    ]
)

# Contar tokens y tipos
registrar_resultado(
    "2. Minúsculas",
    corpus["tokens_minusculas"]
)

# Mostrar un ejemplo
print("\nPrimeros 40 tokens en minúsculas:")
print(corpus["tokens_minusculas"].iloc[0][:40])